# NDT7 (M-Lab) Data Prep — Indonesia Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/id/mlab_id_clean.parquet` (367,247,163 rows — the largest of the
eight countries) into province x quarter format, split into Broadband and Mobile/Cellular,
mirroring the other NDT7 prep notebooks.

**Structural choices — deliberate, mirroring `philippines_ndt7_prep.ipynb`:**

1. **No zoom-16 tile-binning.** The v2 parquet already carries a clean `province` column from a
   GADM point-in-polygon join. `n_tiles` is left as `NaN` and **`is_reliable` uses
   `total_tests >= 100` only** — the same rule now used by every NDT7 country. NDT7 coordinates
   come from MaxMind at *city-centroid* granularity, so a tile count measures how many cities
   MaxMind knows in a province, not how well the data is spread; Ookla keeps `n_tiles >= 5`
   because its coordinates are genuinely spatial. Running tile-binning over 367M rows to produce
   a column nothing reads would be pure cost.
2. **Output granularity is ADM1 province (34).** Indonesia split Papua into new provinces during
   2022–24 and now has 38, but this dataset is pinned at **34**: the parquet was joined with
   GADM 4.1 (2022 boundaries), `data/geo/indonesia_provinces.geojson` is geoBoundaries ADM1
   (2017), and the BPS GRDP table used for the reference CSV is the 2021 edition — all three
   agree on 34. See `data/reference/indonesia_reference_PROVENANCE.md`.
3. **Province names need mapping.** The parquet stores run-together Indonesian names
   (`JawaBarat`, `SumateraUtara`) while the reference CSV and geojson use readable English
   (`West Java`, `North Sumatra`). `PROVINCE_MAP` below covers all 34 — verified against the
   real parquet, not assumed.

**Outputs:**
- `data/exports/ndt7_indonesia_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_indonesia_province_quarterly.csv` — Mobile/Cellular


In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/id/mlab_id_clean.parquet'
ID_REF_CSV  = '../../../data/reference/indonesia_reference.csv'

con = duckdb.connect()
# ไฟล์ 19 GB / 367M แถว — ต้องตั้งไม่งั้น OOM
con.execute("SET memory_limit='2GB'")
con.execute("SET temp_directory='../../../.tmp/duckdb'")   # ที่พักตอน spill
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads=2")  # เครื่องนี้ RAM จำกัด ลด thread ตัดหน่วยความจำซ้อน
print('rows:', con.execute(f"SELECT COUNT(*) FROM read_parquet('{RAW_PARQUET}')").fetchone()[0])

rows: 367247163


### 1. Province x Quarter Aggregation (DuckDB)

รวบงานระดับแถวทั้งหมดไว้ใน query เดียว — กรอง, ติดไตรมาส, group by province x quarter x type x
network_type ไม่มีการ loop ฝั่ง Python

ตัด `hosting` ออกตั้งแต่ใน SQL (เหลือ `broadband`/`cellular`) และตัดแถวที่ `province IS NULL`
(อินโดมี 6,896 แถว = 0.002%)

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")                        # PH/ID ใหญ่ ต้องตั้ง
con.execute("SET temp_directory='../../../.tmp/duckdb'")   # ที่พักตอน spill
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads=2")  # เครื่องนี้ RAM จำกัด ลด thread ตัดหน่วยความจำซ้อน

sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        min_rtt,
        type, network_type, province,
        year,
        CAST(CEIL(month / 3.0) AS INT) AS qtr,
        -- zoom-16 Web Mercator tile — ใช้เป็นคอลัมน์วินิจฉัยเท่านั้น ไม่ได้ใช้คิดค่าเฉลี่ย
        CAST(LEAST(FLOOR((longitude + 180) / 360 * 65536), 65535) AS BIGINT) AS tx,
        CAST(LEAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))
             + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))))) / pi()) / 2 * 65536), 65535) AS BIGINT) AS ty
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND province IS NOT NULL
      AND network_type IN ('broadband', 'cellular')
)
SELECT
    province, network_type, type,
    (CAST(year AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
    AVG(mean_throughput_mbps)                       AS avg_thr,
    AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END)  AS avg_lat,
    COUNT(*)                                        AS test_count,
    COUNT(DISTINCT tx * 65536 + ty) AS n_tiles
FROM filtered
GROUP BY province, network_type, type, year_q
"""

prov_q_all = con.execute(sql).df()
print(f"province x quarter x type x network rows: {len(prov_q_all):,}")
print(f"quarters: {len(prov_q_all['year_q'].unique())} | province: {prov_q_all['province'].nunique()}")
print(prov_q_all.groupby('network_type')['test_count'].sum().apply(lambda x: f'{x:,}'))

province x quarter x type x network rows: 1,544
quarters: 12 | province: 34
network_type
broadband    237,195,963
cellular      93,981,780
Name: test_count, dtype: str


### 2. Province Name Mapping — parquet (`JawaBarat`) → reference (`West Java`)

ทำหลัง aggregate เพราะผลลัพธ์เหลือแค่หลักพันแถว map ด้วย pandas ได้สบาย เหมือนที่ลาวทำ

`assert` ไว้ให้ fail ทันทีถ้ามีชื่อไหนไม่ถูก map — จะได้ไม่เงียบ ๆ แล้วไปโผล่เป็น NaN ตอน join

In [3]:
# parquet เก็บชื่ออินโดนีเซียแบบเขียนติดกัน -> ชื่ออังกฤษอ่านง่ายใน indonesia_reference.csv
# ตรวจกับ parquet จริงแล้ว ครบทั้ง 34 ไม่มีตกหล่น
PROVINCE_MAP = {
    'Aceh': 'Aceh',
    'Bali': 'Bali',
    'BangkaBelitung': 'Bangka Belitung',
    'Banten': 'Banten',
    'Bengkulu': 'Bengkulu',
    'Gorontalo': 'Gorontalo',
    'JakartaRaya': 'Jakarta',
    'Jambi': 'Jambi',
    'JawaBarat': 'West Java',
    'JawaTengah': 'Central Java',
    'JawaTimur': 'East Java',
    'KalimantanBarat': 'West Kalimantan',
    'KalimantanSelatan': 'South Kalimantan',
    'KalimantanTengah': 'Central Kalimantan',
    'KalimantanTimur': 'East Kalimantan',
    'KalimantanUtara': 'North Kalimantan',
    'KepulauanRiau': 'Riau Islands',
    'Lampung': 'Lampung',
    'Maluku': 'Maluku',
    'MalukuUtara': 'North Maluku',
    'NusaTenggaraBarat': 'West Nusa Tenggara',
    'NusaTenggaraTimur': 'East Nusa Tenggara',
    'Papua': 'Papua',
    'PapuaBarat': 'West Papua',
    'Riau': 'Riau',
    'SulawesiBarat': 'West Sulawesi',
    'SulawesiSelatan': 'South Sulawesi',
    'SulawesiTengah': 'Central Sulawesi',
    'SulawesiTenggara': 'Southeast Sulawesi',
    'SulawesiUtara': 'North Sulawesi',
    'SumateraBarat': 'West Sumatra',
    'SumateraSelatan': 'South Sumatra',
    'SumateraUtara': 'North Sumatra',
    'Yogyakarta': 'Yogyakarta',
}

_before = set(prov_q_all['province'])
_unmapped = _before - set(PROVINCE_MAP)
assert not _unmapped, f"มีชื่อจังหวัดที่ยังไม่ได้ map: {_unmapped}"

prov_q_all['province'] = prov_q_all['province'].map(PROVINCE_MAP)
print(f"map แล้ว {len(PROVINCE_MAP)} ชื่อ | เหลือ {prov_q_all['province'].nunique()} จังหวัด")

map แล้ว 34 ชื่อ | เหลือ 34 จังหวัด


### 3. Province-Level Aggregation (per network type) + Reference Merge

กาง download/upload ออกเป็นคอลัมน์ merge ข้อมูลอ้างอิง (GDP/density/tier) แล้วคำนวณ `is_reliable`

**ไม่มี `n_tiles`** — `is_reliable = total_tests >= 100` อย่างเดียว (ดูหัวเล่ม)

In [4]:
def build_province_quarterly(prov_q_all, network_type, ref):
    d = prov_q_all[prov_q_all['network_type'] == network_type]
    print(f"[{network_type}] province x quarter x type rows: {len(d):,}")

    dl = d[d['type'] == 'download'].rename(columns={
        'avg_thr': 'avg_d_mbps', 'avg_lat': 'avg_lat_ms_wt', 'test_count': 'total_tests'})
    ul = d[d['type'] == 'upload'].rename(columns={'avg_thr': 'avg_u_mbps'})

    dl_stats = dl[['year_q', 'province', 'avg_d_mbps', 'avg_lat_ms_wt', 'total_tests', 'n_tiles']]
    ul_stats = ul[['year_q', 'province', 'avg_u_mbps']]

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    # NDT7 ใช้ total_tests อย่างเดียว ไม่ใช้ n_tiles เป็นเกณฑ์ (Ookla ยังใช้ทั้งคู่)
    # เหตุผล: NDT7 ได้พิกัดจาก MaxMind ซึ่งเป็น city centroid ทุก test ในเมืองเดียวกันจึงตกลง tile
    # เดียวกัน n_tiles จึงวัด "จังหวัดนี้มีกี่เมืองใน MaxMind" ไม่ได้วัดการกระจายตัวของข้อมูล
    # (ลาวทั้งประเทศมีพิกัดต่างกัน 33 จุด n_tiles สูงสุด = 3 -> เกณฑ์ >=5 เป็นไปไม่ได้)
    # คอลัมน์ n_tiles ยังเก็บไว้ให้ดูใน "Data Quality" ของ EDA
    master['is_reliable'] = master['total_tests'] >= 100
    print(f"[{network_type}] province x quarter rows: {len(master)} | "
          f"reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

ref = pd.read_csv(ID_REF_CSV)
print(f"reference: {len(ref)} แถว")


reference: 34 แถว


---
## Part 1 — Broadband

In [5]:
broadband_master = build_province_quarterly(prov_q_all, 'broadband', ref)
broadband_master.head()

[broadband] province x quarter x type rows: 808
[broadband] province x quarter rows: 404 | reliable: 402 (99.5%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Aceh,15.747724,75.899669,63239,13,6.033186,2023,1,True,Sumatra,4,5626000,2423.0,99,7298.0,233536.0
1,2023-Q1,Bali,15.904471,73.409516,151732,24,10.354946,2023,1,True,Lesser Sunda Islands,3,4461300,3521.0,799,10605.0,339360.0
2,2023-Q1,Bangka Belitung,10.661013,76.923891,2478,2,7.067898,2023,1,True,Sumatra,2,1550800,4077.0,93,12280.0,392960.0
3,2023-Q1,Banten,15.876682,61.176110,407542,56,8.391010,2023,1,True,Java,2,12537400,3859.0,1340,11623.0,371936.0
4,2023-Q1,Bengkulu,17.454716,65.458867,42762,3,6.927284,2023,1,True,Sumatra,4,2138000,2736.0,106,8241.0,263712.0


In [6]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_indonesia_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 404 rows -> ../../../data/exports/ndt7_indonesia_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Aceh,2023-Q1,2023,1,15.747724,6.033186,75.899669,63239,13,True,Sumatra,4,5626000,2423.0,99,7298.0,233536.0
1,Bali,2023-Q1,2023,1,15.904471,10.354946,73.409516,151732,24,True,Lesser Sunda Islands,3,4461300,3521.0,799,10605.0,339360.0
2,Bangka Belitung,2023-Q1,2023,1,10.661013,7.067898,76.923891,2478,2,True,Sumatra,2,1550800,4077.0,93,12280.0,392960.0


---
## Part 2 — Mobile/Cellular

In [7]:
mobile_master = build_province_quarterly(prov_q_all, 'cellular', ref)
mobile_master.head()

[cellular] province x quarter x type rows: 736
[cellular] province x quarter rows: 368 | reliable: 361 (98.1%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Aceh,22.578934,62.443540,10625,4,6.603546,2023,1,True,Sumatra,4,5626000,2423.0,99,7298.0,233536.0
1,2023-Q1,Bali,16.390216,65.424790,55358,6,8.890478,2023,1,True,Lesser Sunda Islands,3,4461300,3521.0,799,10605.0,339360.0
2,2023-Q1,Banten,13.837379,80.049349,37394,8,8.292109,2023,1,True,Java,2,12537400,3859.0,1340,11623.0,371936.0
3,2023-Q1,Bengkulu,10.187513,92.878351,3957,1,6.603398,2023,1,True,Sumatra,4,2138000,2736.0,106,8241.0,263712.0
4,2023-Q1,Central Java,12.119875,73.227863,399728,15,7.290826,2023,1,True,Java,4,38233900,2703.0,1113,8141.0,260512.0


In [8]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_indonesia_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 368 rows -> ../../../data/exports/ndt7_mobile_indonesia_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Aceh,2023-Q1,2023,1,22.578934,6.603546,62.443540,10625,4,True,Sumatra,4,5626000,2423.0,99,7298.0,233536.0
1,Bali,2023-Q1,2023,1,16.390216,8.890478,65.424790,55358,6,True,Lesser Sunda Islands,3,4461300,3521.0,799,10605.0,339360.0
2,Banten,2023-Q1,2023,1,13.837379,8.292109,80.049349,37394,8,True,Java,2,12537400,3859.0,1340,11623.0,371936.0


## Summary

- **Input:** `data/ndt7/id/mlab_id_clean.parquet` — 367,247,163 rows, ISP-classified (v2) and
  province-joined. ไฟล์ใหญ่ที่สุดในชุด (19 GB)
- **Output:** province x quarter aggregates, 34 ADM1 provinces, Broadband และ Mobile แยกกัน
- **Reliability:** `total_tests >= 100` เท่านั้น — ไม่มี `n_tiles` เหมือนทุกประเทศ NDT7 ตอนนี้
  (Ookla ยังใช้ `n_tiles >= 5` อยู่ อย่าเอาไปเทียบกันตรง ๆ)
- **ข้อควรระวังของอินโดโดยเฉพาะ** (ดู `data/reference/indonesia_reference_PROVENANCE.md`):
  - `pop_2024` จริง ๆ เป็นตัวเลข **mid-2025** จาก BPS — คงชื่อคอลัมน์ไว้เพื่อให้ notebook อื่นไม่พัง
  - `internet_tier` เป็น **ควอร์ไทล์ GDP ตรง ๆ ไม่มีการปรับมือ** ต่างจากของไทย → ห้ามเทียบ tier ข้ามประเทศ
  - **34 จังหวัด ไม่ใช่ 38** — ยุบปาปัวใหม่กลับเข้า Papua/West Papua ให้ตรง GADM 4.1 (2022)
  - **Jakarta กิน test ไป ~124M จาก 331M (37%)** — MaxMind ลงพิกัดเป็น centroid เมือง
    ค่าเฉลี่ยระดับประเทศจึงเอนไปทางจาการ์ตาหนักมาก ต้องระวังตอนตีความ
